# HireSense - Resume Parser & Experience Validator

This notebook implements the resume parsing layer. It satisfies the following requirements:
- Extracts only four sections: **SKILLS**, **EXPERIENCE**, **PROJECTS**, and **EDUCATION**.
- Handles header synonyms case-insensitively.
- Enforces the **Critical Section Isolation Rule** to avoid cross-contamination.
- Enforces the **Experience Validation Rule**: checks if job role, company, and duration exist in experience entries. Otherwise sets Experience Score = 0.
- Detects and rejects scanned or image-only PDFs, returning a user-friendly error message.

In [1]:
import os
import re
import json
import pdfplumber
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Resume Parser Implementation

We implement the parser code here. We will use a regex search to identify header start/end character offsets, sort them, slice the text into isolated regions, and parse each region.

In [ ]:
# Header regex dictionary
HEADERS_REGEX = {
    'SKILLS': re.compile(r'^\s*[-*•]?\s*(skills|technical\s+skills|core\s+skills)\s*:?\s*$', re.IGNORECASE | re.MULTILINE),
    'EXPERIENCE': re.compile(r'^\s*[-*•]?\s*(experience|work\s+experience|professional\s+experience|internships)\s*:?\s*$', re.IGNORECASE | re.MULTILINE),
    'PROJECTS': re.compile(r'^\s*[-*•]?\s*(projects|key\s+projects|academic\s+projects|personal\s+projects|notable\s+projects)\s*:?\s*$', ...
    'EDUCATION': re.compile(r'^\s*[-*•]?\s*(education|academic\s+background)\s*:?\s*$', re.IGNORECASE | re.MULTILINE)
}

def extract_text_from_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            return text
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
        return None

def validate_experience_entry(entry_text):
    """
    Checks if an experience entry contains:
    1. Job Role
    2. Company/organization
    3. Duration
    """
    role_pattern = re.compile(r'(engineer|developer|scientist|manager|intern|analyst|programmer|lead|specialist|architect|coordinator|designer)', re.IGNORECASE)
    company_pattern = re.compile(r'(google|microsoft|meta|amazon|netflix|apple|stripe|uber|airbnb|salesforce|atlassian|slack|technologies|solutions|labs|corp|inc|llc|co\b|global|tech|group|systems|enterprise|ventures)', re.IGNORECASE)
    # Matches common date range formats like "June 2025 - August 2025", "Jan 2022 - Present", "2020 - 2022"
    duration_pattern = re.compile(r'((?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s*\d{4}\s*(?:-|–|—|to)\s*(?:present|(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s*\d{4})|\d{4}\s*(?:-|–|—|to)\s*(?:\d{4}|present))', re.IGNORECASE)
    
    has_role = bool(role_pattern.search(entry_text))
    has_company = bool(company_pattern.search(entry_text))
    has_duration = bool(duration_pattern.search(entry_text))
    
    return has_role and has_company and has_duration

def parse_resume(pdf_path):
    text = extract_text_from_pdf(pdf_path)
    
    # Scanned PDF check
    if not text or len(text.strip()) < 100:
        return {
            "success": False,
            "error": "Scanned or image-only PDF detected. Please upload a text-based PDF resume."
        }
    
    # Find all header matches
    matches = []
    for section, regex in HEADERS_REGEX.items():
        for m in regex.finditer(text):
            matches.append({
                'section': section,
                'start': m.start(),
                'end': m.end(),
                'line': m.group(0)
            })
            
    # Sort matches by start index to isolate sections
    matches = sorted(matches, key=lambda x: x['start'])
    
    parsed_sections = {
        'SKILLS': '',
        'EXPERIENCE': '',
        'PROJECTS': '',
        'EDUCATION': ''
    }
    
    for i, m in enumerate(matches):
        start_idx = m['end']
        # The section content goes until the start of the next section header
        end_idx = matches[i+1]['start'] if i + 1 < len(matches) else len(text)
        
        section_text = text[start_idx:end_idx].strip()
        # Store only if we don't already have text (keep first match or append if multiple)
        if parsed_sections[m['section']]:
            parsed_sections[m['section']] += "\n" + section_text
        else:
            parsed_sections[m['section']] = section_text
            
    # Validate Experience Section
    exp_text = parsed_sections['EXPERIENCE'].strip()
    is_exp_valid = False
    if exp_text:
        # Split by paragraph to check individual jobs
        entries = [e.strip() for e in exp_text.split('\n\n') if e.strip()]
        for entry in entries:
            if validate_experience_entry(entry):
                is_exp_valid = True
                break
                
    parsed_sections['EXPERIENCE_VALID'] = is_exp_valid
    
    return {
        "success": True,
        "data": parsed_sections
    }

## 2. Testing the Parser on Synthetic Data

We run the parser across all 205 generated PDF resumes to check the parsing rate and validation flags.

In [3]:
results = []
for idx in range(1, 206):
    path = f"../data/resumes/resume_{idx:03d}.pdf"
    parsed = parse_resume(path)
    results.append({
        "id": idx,
        "filename": f"resume_{idx:03d}.pdf",
        "success": parsed["success"],
        "error": parsed.get("error", None),
        "skills_extracted": len(parsed.get("data", {}).get("SKILLS", "")) > 0 if parsed["success"] else False,
        "experience_valid": parsed.get("data", {}).get("EXPERIENCE_VALID", False) if parsed["success"] else False,
        "education_extracted": len(parsed.get("data", {}).get("EDUCATION", "")) > 0 if parsed["success"] else False
    })

df = pd.DataFrame(results)
print(df.tail(15))

      id        filename  success  \
190  191  resume_191.pdf     True   
191  192  resume_192.pdf     True   
192  193  resume_193.pdf     True   
193  194  resume_194.pdf     True   
194  195  resume_195.pdf     True   
195  196  resume_196.pdf     True   
196  197  resume_197.pdf     True   
197  198  resume_198.pdf     True   
198  199  resume_199.pdf     True   
199  200  resume_200.pdf     True   
200  201  resume_201.pdf    False   
201  202  resume_202.pdf    False   
202  203  resume_203.pdf    False   
203  204  resume_204.pdf    False   
204  205  resume_205.pdf    False   

                                                 error  skills_extracted  \
190                                                NaN              True   
191                                                NaN              True   
192                                                NaN              True   
193                                                NaN              True   
194                        

## 3. Analysis & Visualizations of Parsing Results

In [4]:
print("Success Rate:", df["success"].mean() * 100, "%")
print("Experience Validation Rate (of successful parses):", df[df["success"]]["experience_valid"].mean() * 100, "%")
print("Rejection Count:", df[~df["success"]].shape[0])

Success Rate: 97.5609756097561 %
Experience Validation Rate (of successful parses): 94.5 %
Rejection Count: 5
